# Hypothesis Testing

Make rigorous decisions from data — from A/B tests to model comparisons, with proper power analysis and multiple testing correction.

**Topics:** Null/alternative hypotheses, p-values, Type I/II errors, t-tests, chi-square, Mann-Whitney, power analysis, multiple comparisons

## 1. Framework — Hypotheses, p-values, Error Types

In [ ]:
import numpy as np
from scipy import stats

# The hypothesis testing framework
framework = {
    'H0 (Null)':        'The default assumption. "No effect, no difference."',
    'H1 (Alternative)': 'What you are trying to prove. "There is an effect."',
    'p-value':          'P(seeing data this extreme | H0 is true). NOT P(H0 is true)!',
    'α (alpha)':        'Threshold for rejection. Typically 0.05. Set BEFORE looking at data.',
    'Reject H0 if':     'p-value < α',
}
print('Hypothesis testing framework:')
for key, desc in framework.items():
    print(f'  {key:<20}: {desc}')
print()

# Error types
print('Decision matrix:')
print(f'{"":<25} {"H0 TRUE":<25} {"H0 FALSE"}')
print(f'{"Reject H0 (claim effect)":<25} {"Type I Error (FP) = α":<25} {"Correct! = Power = 1-β"}')
print(f'{"Fail to reject H0":<25} {"Correct! = 1-α":<25} {"Type II Error (FN) = β"}')
print()
print('Type I Error (False Positive): saying effect exists when it does not')
print('  → Controlled by α. Typically α = 0.05 means 5% chance of false alarm.')
print()
print('Type II Error (False Negative): missing a real effect')
print('  → Controlled by β. Power = 1 - β. Typically target Power = 0.80.')
print()

# Common misconceptions about p-values
misconceptions = [
    ('WRONG: p=0.05 means 95% chance H1 is true', 'p-value says nothing about P(H0) or P(H1)'),
    ('WRONG: p=0.001 means strong effect', 'Small p with large n can mean tiny, meaningless effect'),
    ('WRONG: p=0.06 means no effect', 'It means insufficient evidence, not absence of effect'),
]
print('Common p-value misconceptions:')
for wrong, right in misconceptions:
    print(f'  {wrong}')
    print(f'  → {right}')
    print()

## 2. t-Tests — Comparing Means

In [ ]:
from scipy import stats
import numpy as np

np.random.seed(42)

# One-sample t-test: is our sample mean = claimed population mean?
# Scenario: company claims avg delivery time = 3 days. We observe:
delivery_times = np.random.normal(3.4, 0.8, 50)  # our sample
t_stat, p_val = stats.ttest_1samp(delivery_times, popmean=3.0)
ci = stats.t.interval(0.95, df=len(delivery_times)-1, loc=delivery_times.mean(), scale=stats.sem(delivery_times))

print('One-sample t-test: Is mean delivery time = 3 days?')
print(f'  Sample mean: {delivery_times.mean():.2f} days (n={len(delivery_times)})')
print(f'  t={t_stat:.3f}, p={p_val:.4f}')
print(f'  95% CI: [{ci[0]:.2f}, {ci[1]:.2f}] days')
print(f'  Decision: {"REJECT H0" if p_val < 0.05 else "FAIL TO REJECT H0"} — delivery is {"slower" if p_val < 0.05 else "consistent with"} claimed 3 days')
print()

# Two-sample t-test: A/B test comparison
control = np.random.normal(12.5, 2.0, 200)   # avg order value, control
treatment = np.random.normal(13.2, 2.1, 200)  # avg order value, new UI

# Welch's t-test (don't assume equal variances — safer)
t_stat, p_val = stats.ttest_ind(control, treatment, equal_var=False)
effect_size = (treatment.mean() - control.mean()) / np.sqrt((control.std()**2 + treatment.std()**2) / 2)

print('Two-sample t-test: A/B test — new UI effect on order value')
print(f'  Control:   mean=${control.mean():.2f}, n={len(control)}')
print(f'  Treatment: mean=${treatment.mean():.2f}, n={len(treatment)}')
print(f'  Lift: ${treatment.mean() - control.mean():.2f} ({(treatment.mean()/control.mean()-1)*100:.1f}%)')
print(f'  t={t_stat:.3f}, p={p_val:.4f}')
print(f'  Cohen\'s d = {effect_size:.3f} ({"small" if abs(effect_size) < 0.5 else "medium" if abs(effect_size) < 0.8 else "large"} effect)')
print(f'  Decision: {"REJECT H0 — new UI is better" if p_val < 0.05 else "FAIL TO REJECT H0"}')

## 3. Non-Parametric Tests — When Normality Fails

In [ ]:
from scipy import stats
import numpy as np

np.random.seed(42)

# Check normality first
non_normal_data = stats.expon.rvs(scale=10, size=50)  # exponential distribution
normal_data = np.random.normal(10, 3, 50)

print('Normality tests:')
for name, data in [('Non-normal (Exponential)', non_normal_data), ('Normal', normal_data)]:
    shapiro_stat, shapiro_p = stats.shapiro(data)
    print(f'  {name}: Shapiro-Wilk p={shapiro_p:.4f} → {"NOT normal" if shapiro_p < 0.05 else "Consistent with normal"}')
print()

# Mann-Whitney U: non-parametric alternative to 2-sample t-test
group_a = stats.expon.rvs(scale=10, size=100)   # support response time A
group_b = stats.expon.rvs(scale=12, size=100)   # support response time B (slower)

# t-test (wrong: assumes normality)
t_stat, t_p = stats.ttest_ind(group_a, group_b)
# Mann-Whitney (correct: rank-based)
mw_stat, mw_p = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')

print('Comparing response times (non-normal data):')
print(f'  Group A: median={np.median(group_a):.1f}s, Group B: median={np.median(group_b):.1f}s')
print(f'  t-test:        p={t_p:.4f}')
print(f'  Mann-Whitney:  p={mw_p:.4f}  ← more appropriate here')
print()

# Chi-square: testing independence of categorical variables
# Scenario: Does button color affect conversion?
observed = np.array([[200, 50], [180, 70]])  # [[blue_conv, blue_no], [red_conv, red_no]]
chi2, chi2_p, dof, expected = stats.chi2_contingency(observed)

print('Chi-square test: Button color vs conversion')
print(f'  Blue: {observed[0,0]} convert, {observed[0,1]} do not')
print(f'  Red:  {observed[1,0]} convert, {observed[1,1]} do not')
print(f'  Chi2={chi2:.3f}, p={chi2_p:.4f}')
print(f'  Decision: {"Button color matters!" if chi2_p < 0.05 else "No significant effect"}')
print()
print('Test selection guide:')
tests = [
    ('1 group, continuous, normal', 'One-sample t-test'),
    ('2 groups, continuous, normal', 'Two-sample t-test (Welch)'),
    ('2 groups, continuous, non-normal', 'Mann-Whitney U'),
    ('2+ groups, continuous, normal', 'One-way ANOVA'),
    ('2 categorical variables', 'Chi-square test'),
    ('Paired observations', 'Paired t-test or Wilcoxon signed-rank'),
]
for situation, test in tests:
    print(f'  {situation:<40}: {test}')

## 4. Statistical Power and Sample Size

In [ ]:
from scipy import stats
import numpy as np

def power_analysis(
    effect_size: float,  # Cohen's d
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """Calculate required sample size per group for a two-sample t-test."""
    # Analytical solution
    z_alpha = stats.norm.ppf(1 - alpha / 2)  # two-tailed
    z_beta = stats.norm.ppf(power)
    n = ((z_alpha + z_beta) / effect_size) ** 2 * 2
    return int(np.ceil(n))

def empirical_power(n: int, effect_size: float, alpha: float = 0.05, n_sim: int = 5000) -> float:
    """Simulate power: fraction of experiments correctly rejecting H0."""
    rejections = 0
    for _ in range(n_sim):
        control = np.random.normal(0, 1, n)
        treatment = np.random.normal(effect_size, 1, n)
        _, p = stats.ttest_ind(control, treatment)
        if p < alpha:
            rejections += 1
    return rejections / n_sim

# Sample size requirements
print('Sample size per group (80% power, α=0.05):')
print(f'{"Effect size":<15} {"Cohen\'s d":<12} {"n per group":<14} {"Scenario"}')
print('-' * 65)
scenarios = [
    ('Very small', 0.1, 'Detecting 0.1% CTR lift in large-scale A/B test'),
    ('Small', 0.2, 'Detecting 1% revenue improvement'),
    ('Medium', 0.5, 'Detecting 5% conversion improvement'),
    ('Large', 0.8, 'Detecting major feature impact'),
    ('Very large', 1.2, 'Detecting completely different experiences'),
]
for name, d, scenario in scenarios:
    n = power_analysis(d)
    print(f'{name:<15} {d:<12} {n:<14} {scenario}')

print()
print('Power grows with: larger n, larger effect, higher α')
print('Key insight: running underpowered experiments wastes time — either real effects are missed')
print('or you\'d need to re-run with more data anyway.')

## 5. Multiple Testing Correction

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

# The multiple testing problem
n_tests = 20
alpha = 0.05
fwer = 1 - (1 - alpha) ** n_tests  # Family-Wise Error Rate

print(f'Multiple testing problem:')
print(f'  {n_tests} tests at α={alpha} each')
print(f'  P(at least one false positive) = 1 - (1-α)^k = {fwer:.1%}')
print(f'  → Without correction, you\'ll get ~{n_tests*alpha:.0f} false positives per 20 tests!')
print()

# Simulate: 20 tests, all H0 true (no real effects)
np.random.seed(42)
null_pvalues = np.random.uniform(0, 1, n_tests)  # under H0, p-values are uniform
# Inject 3 truly significant p-values
null_pvalues[:3] = np.array([0.001, 0.003, 0.02])  # real effects

print('Correction methods compared:')
correction_methods = [
    ('Bonferroni', 'bonferroni', 'Strictest — controls FWER. α/k per test'),
    ('Holm-Bonferroni', 'holm', 'Less strict than Bonferroni, still FWER'),
    ('Benjamini-Hochberg', 'fdr_bh', 'Controls FDR — recommended for discovery'),
    ('Benjamini-Yekutieli', 'fdr_by', 'FDR control under dependence'),
]
print(f'{"Method":<25} {"Significant tests found":<25} {"Type"}')
print('-' * 75)
for name, method, desc in correction_methods:
    reject, p_corrected, _, _ = multipletests(null_pvalues, alpha=0.05, method=method)
    print(f'{name:<25} {reject.sum():<25} {desc}')

print()
print('FWER (Family-Wise Error Rate): P(any false positive) < α  ← strict, use for confirmatory')
print('FDR (False Discovery Rate): E(false positives / all positives) < α  ← lenient, use for exploration')
print()
print('Practical rule: Use Bonferroni for <5 tests. BH-FDR for genomics/feature selection.')

## Exercises

1. **A/B test design:** You want to detect a 2% lift in conversion rate (from 5% to 7%). Calculate required sample size per group. Simulate 1000 experiments at this sample size — what fraction correctly detect the lift? What happens if you stop early at 50% of the planned sample size?

2. **Multiple testing simulation:** Run 100 hypothesis tests where H0 is always true (all null). Count false positives with no correction, Bonferroni, and BH-FDR. Repeat 1000 times. Plot the distribution of false positive counts for each method.

3. **Bootstrapped confidence interval:** Implement bootstrap confidence intervals for the median (not just the mean). Compare to the analytical formula. Show they agree for large n but diverge for n=20 on skewed data.